# 📖 Bible Reading Progress Tracker — NER BERT Fine-tuning

**Purpose**  
Fine-tune IndoBERT for Named Entity Recognition (NER) on Bible reading progress messages.

| # | Section | Description |
|---|---------|-------------|
| 1 | **Setup** | Experiment config, imports |
| 2 | **Model & Tokenizer** | Load IndoBERT base + label schema |
| 3 | **Data** | Load CoNLL, split, tokenize |
| 4 | **Training** | Fine-tune with `Trainer` |
| 5 | **Evaluation** | Metrics + save model |
| 6 | **Inference** | Sanity-check pipeline on test sentences |

## 1. Setup

### 1.1 Experiment Config

In [37]:
import sys
from pathlib import Path

sys.path.append(str(Path('..').resolve() / 'src'))

from config.settings import PROCESSED_DIR, CONFIG_PATH, MODEL_DIR

# ── Experiment Config ──────────────────────────────────────────────────────
EXPERIMENT_NAME = "indobert-bible-ner-v6"
DATA_PATH  = PROCESSED_DIR / "NER_tasks" / "ner_tasks_1700.conll"
SAVE_PATH = MODEL_DIR / EXPERIMENT_NAME
OUTPUT_DIR = f'./runs/{EXPERIMENT_NAME}'

# ── Training knobs ───────────────────────────────────────────────────────────
NUM_EPOCHS = 30 

print(f'Experiment      : {EXPERIMENT_NAME}')
print(f'Data            : {DATA_PATH}')
print(f'Max epochs      : {NUM_EPOCHS}')
print(f'Save path       : {SAVE_PATH}')

Experiment      : indobert-bible-ner-v6
Data            : C:\one one\Desktop\bible_reading_recap_nlp\data\processed\NER_tasks\ner_tasks_1700.conll
Max epochs      : 30
Save path       : C:\one one\Desktop\bible_reading_recap_nlp\models\indobert-bible-ner-v6


### 1.2 Imports

In [2]:
import warnings
import numpy as np

import evaluate
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    EarlyStoppingCallback,
    DataCollatorForTokenClassification,
    TrainingArguments, 
    Trainer, 
    pipeline
)

warnings.filterwarnings('ignore')

from bible_data import load_bible_data
from extraction.extractor import load_ner_model, parse_ner_response
from preprocessing.normalization.normalizer import BibleReferenceNormalizer

normalizer = BibleReferenceNormalizer()

print('Setup complete.')

c:\one one\Desktop\bible_reading_recap_nlp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete.


## 2. Model & Tokenizer

**Base model:** IndoBERT — pre-trained on Indonesian text, which gives it implicit knowledge of Indonesian Bible book names even before fine-tuning.

**Label schema** — BIO tagging so `seqeval` evaluates entity boundaries correctly:

| Label | Meaning |
|-------|---------|
| `O` | Outside any entity |
| `B-BOOK` | First token of a book name |
| `I-BOOK` | Continuation token of a book name |
| `B-CHAPTER` | First token of a chapter number |
| `I-CHAPTER` | Continuation token of a chapter number |

In [3]:
model, tokenizer = load_ner_model(CONFIG_PATH)

bible_data = load_bible_data()

ORIGINAL_VOCAB_SIZE = len(tokenizer.get_vocab())

# Compound book names that WordPiece splits incorrectly
compound_tokens = [
    book['name']
    for book in bible_data['books'] if '-' in book['name']
]

existing = set(tokenizer.get_vocab().keys())
new_tokens = [t for t in compound_tokens if t not in existing]

tokenizer.add_tokens(new_tokens)
print(f'Added {len(new_tokens)} tokens: {new_tokens}')
print(f'Vocab size: {len(tokenizer)}')

2026-04-05 10:05:42 | INFO     | bible_pipeline.extraction.extractor | Loading pretrained model: indolem/indobert-base-uncased


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 461.22it/s, Materializing param=bert.encoder.layer.11.output.dense.weight]              
BertForTokenClassification LOAD REPORT from: indolem/indobert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNE

Added 3 tokens: ['Hakim-hakim', '1 Raja-raja', '2 Raja-raja']
Vocab size: 31926


In [4]:
# Sanity-check: verify compound tokens are now single ids
for tok in new_tokens:
    ids = tokenizer.convert_tokens_to_ids([tok])
    pieces = tokenizer.tokenize(tok)
    status = '✓' if len(pieces) == 1 else '✗ still splits'
    print(f'  {tok!r:20s} → id={ids[0]}  pieces={pieces}  {status}')

  'Hakim-hakim'        → id=31923  pieces=['hakim-hakim']  ✓
  '1 Raja-raja'        → id=31924  pieces=['1 raja-raja']  ✓
  '2 Raja-raja'        → id=31925  pieces=['2 raja-raja']  ✓


In [5]:
model.resize_token_embeddings(len(tokenizer))

id2label =  model.config.id2label
label2id = model.config.label2id
print("Label schema", id2label)

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Label schema {0: 'O', 1: 'B-BIBLE_REF', 2: 'I-BIBLE_REF'}


In [6]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Trainable params : {trainable_params:,}')

Trainable params : 109,972,227


## 3. Data

### 3.1 Load CoNLL

CoNLL format - one token per line, blank lines seperate sentences.

```
Efesus  BOOK
1       CHAPTER
-       O
2       CHAPTER
done    O
```

In [7]:
def read_conll(filepath: Path):
    """
    Parse a CoNLL-format file into (sentences, labels) lists.
    """
    sentences, labels, raw_texts = [], [], []
    tokens, ner_tags = [], []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                if tokens:
                    sentences.append(tokens)
                    labels.append(ner_tags)
                    raw_texts.append(' '.join(tokens))
                    tokens, ner_tags = [], []
            else:
                parts = line.split()
                if parts[0] == "-DOCSTART-":
                    continue

                tokens.append(parts[0])
                ner_tags.append(parts[-1])
    if tokens:
        sentences.append(tokens)
        labels.append(ner_tags)
        raw_texts.append(' '.join(tokens))

    return sentences, labels, raw_texts

In [8]:
import re
import ast
import emoji
import unicodedata
from typing import Any

CROSS_CHAPTER_VERSE_RE = re.compile(
    r'\d+:\d+\s*[-\u2013\u2014]{1,2}\s*\d+:\d+'
    r'|\d+:\d+\s+(?:sampai|sampe|hingga|to)\s+\d+:\d+',
    re.IGNORECASE,
)
VERSE_RANGE_RE = re.compile(
    r'\d+:\d+\s*[-\u2013\u2014]{1,2}\s*\d+(?!\s*:)'
    r'|\d+\s*[-\u2013\u2014]{1,2}\s*\d+:\d+'
    r'|\d+:\d+\s+(?:sampai|sampe|hingga|to)\s+\d+(?!\s*:)',
    re.IGNORECASE,
)
SINGLE_VERSE_RE = re.compile(r'\d+:\d+')
CROSS_BOOK_RE = re.compile(
    r'\d+\s*[-\u2013\u2014]{1,2}\s*[A-Za-z]'
    r'|\d+\s+(?:sampai|sampe|hingga|to)\s+[A-Za-z]',
    re.IGNORECASE,
)
CHAPTER_RANGE_RE = re.compile(
    r'\d+\s*[-\u2013\u2014]{1,2}\s*\d+'
    r'|\d+\s+(?:sampai|sampe|hingga|to)\s+\d+',
    re.IGNORECASE,
)
MARKDOWN_RE = re.compile(r'[_*]')


def is_noisy(msg: str) -> bool:
    has_emoji = any(unicodedata.category(c) in ('So', 'Sm') or c in emoji.EMOJI_DATA for c in msg)
    has_markdown = bool(MARKDOWN_RE.search(msg))
    return has_emoji or has_markdown


def parse_spans(spans: Any) -> list[dict]:
    if isinstance(spans, str):
        return ast.literal_eval(spans)
    return spans or []

def detect_pattern_from_conll(tokens: list[str], ner_tags: list[str]) -> str:
    """Infer pattern pool from token list + BIO labels."""
    text     = ' '.join(tokens)
    ref_toks = [t for t, l in zip(tokens, ner_tags) if l != 'O']
    ref_text = ' '.join(ref_toks)

    if is_noisy(text):
        return 'noisy'

    if not ref_toks:
        return 'none'

    # Count distinct B- spans → multi if ≥ 2
    b_count = sum(1 for l in ner_tags if l == 'B-BIBLE_REF')
    if b_count >= 2:
        return 'multi'

    # Classify the single span by its text
    if CROSS_CHAPTER_VERSE_RE.search(ref_text):
        return 'cross_chapter_verse'
    if VERSE_RANGE_RE.search(ref_text):
        return 'verse_range'
    if SINGLE_VERSE_RE.search(ref_text):
        return 'single_verse'
    if CROSS_BOOK_RE.search(ref_text):
        return 'cross_book'
    if CHAPTER_RANGE_RE.search(ref_text):
        return 'chapter_range'
    if re.search(r'\d', ref_text):
        return 'single_chapter'
    return 'book_only'

In [9]:
sentences, labels, raw_texts = read_conll(DATA_PATH)

pattern_labels = [
    detect_pattern_from_conll(toks, tags)
    for toks, tags in zip(sentences, labels)
]

from collections import Counter
print("Pattern distribution in dataset:")
for pat, n in sorted(Counter(pattern_labels).items()):
    print(f"  {pat:22s}: {n}")

Pattern distribution in dataset:
  book_only             : 127
  chapter_range         : 345
  cross_book            : 276
  cross_chapter_verse   : 17
  multi                 : 279
  noisy                 : 315
  none                  : 106
  single_chapter        : 189
  single_verse          : 8
  verse_range           : 43


### 3.2 Train / Eval Split

80/20 stratified by label presence — ensures both splits see `B-BOOK`, `I-BOOK`, `B-CHAPTER`, `I-CHAPTER` examples.

In [10]:
(train_sent, eval_sent,
 train_labels, eval_labels,
 train_patterns, eval_patterns) = train_test_split(
    sentences, labels, pattern_labels,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=pattern_labels,
)

print(f'Train : {len(train_sent)} | Eval : {len(eval_sent)}')
print(f'\nEval pattern breakdown:')
for pat, n in sorted(Counter(eval_patterns).items()):
    print(f'  {pat:22s}: {n}')

Train : 1364 | Eval : 341

Eval pattern breakdown:
  book_only             : 25
  chapter_range         : 69
  cross_book            : 55
  cross_chapter_verse   : 3
  multi                 : 56
  noisy                 : 63
  none                  : 21
  single_chapter        : 38
  single_verse          : 2
  verse_range           : 9


### 3.3 Tokenize & Align Labels

IndoBERT uses WordPiece tokenization — a single word like `Korintus` may split into multiple subword tokens.  
Only the **first subword** of each word gets the real label; continuation subwords are masked with `-100` so they are ignored by the loss.

In [11]:
raw_dataset = DatasetDict({
    'train': Dataset.from_dict({
        'tokens': train_sent,
        'ner_tags': train_labels,
        'pattern': train_patterns,
    }),
    'eval': Dataset.from_dict({
        'tokens': eval_sent,
        'ner_tags': eval_labels,
        'pattern': eval_patterns,
    }),
})

In [12]:
def tokenize_and_align_labels(example):
    """
    Tokenise a pre-split sentence and propagate NER labels to subword tokens.
    Continuation tokens (word_idx == previous_word_idx) are masked with -100
    so they are ignored by the loss.
    """
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        max_length=512,
        is_split_into_words=True,
    )

    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None
    label_ids = []

    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(-100)
        elif word_idx != previous_word_idx:
            label_ids.append(label2id[example["ner_tags"][word_idx]])
        else:
            label_ids.append(-100)

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = label_ids
    return tokenized_inputs

In [13]:
tokenized_dataset = raw_dataset.map(tokenize_and_align_labels, batched=False)
tokenized_dataset = tokenized_dataset.remove_columns(["tokens", "ner_tags"])
tokenized_dataset.set_format("torch")
tokenized_dataset

Map: 100%|██████████| 341/341 [00:00<00:00, 3984.28 examples/s]


DatasetDict({
    train: Dataset({
        features: ['pattern', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1364
    })
    eval: Dataset({
        features: ['pattern', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 341
    })
})

## 4. Training

### 4.1 Metrics
Using `seqeval` for entity-level evaluation.

In [14]:
def make_compute_metrics(id2label, eval_patterns):
    seqeval = evaluate.load('seqeval')

    def compute_metrics(eval_preds):
        logits, labels = eval_preds
        predictions    = np.argmax(logits, axis=-1)

        true_labels, true_preds = [], []
        for pred_seq, label_seq in zip(predictions, labels):
            cur_l, cur_p = [], []
            for pred_id, label_id in zip(pred_seq, label_seq):
                if label_id != -100:
                    cur_l.append(id2label[label_id])
                    cur_p.append(id2label[pred_id])
            true_labels.append(cur_l)
            true_preds.append(cur_p)

        overall = seqeval.compute(predictions=true_preds, references=true_labels)

        per_pattern = {}
        for pat in set(eval_patterns):
            idx = [i for i, p in enumerate(eval_patterns) if p == pat]
            if len(idx) < 2:
                continue
            r = seqeval.compute(
                predictions=[true_preds[i] for i in idx],
                references=[true_labels[i] for i in idx],
            )
            per_pattern[f'f1_{pat}'] = round(r['overall_f1'], 4)

        return {
            'precision': overall['overall_precision'],
            'recall': overall['overall_recall'],
            'f1': overall['overall_f1'],
            'accuracy': overall['overall_accuracy'],
            **per_pattern,
        }

    return compute_metrics

### 4.2 Training Arguments

In [15]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    seed=42,

    # Optimiser
    learning_rate=3e-5,
    label_smoothing_factor=0.1,
    weight_decay=0.01,
    warmup_steps=int(0.1 * (len(train_sent) / 8) * NUM_EPOCHS),

    # Batch / accumulation
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,

    # Schedule
    lr_scheduler_type="cosine",
    num_train_epochs=NUM_EPOCHS,
    max_grad_norm=1.0,

    # Checkpoint
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    save_total_limit=3,
    logging_steps=20,
)

### 4.3 Train

In [16]:
compute_metrics = make_compute_metrics(id2label, eval_patterns)

data_collator = DataCollatorForTokenClassification(tokenizer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["eval"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(
        early_stopping_patience=5,
        early_stopping_threshold=0.001
        )]
)

trainer.train()

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "C:\Python312\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "C:\Python312\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "c:\one one\Desktop\bible_reading_recap_nlp\venv\Lib\site-packages\transformers\safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "c:\one one\Desktop\bible_reading_recap_nlp\venv\Lib\site-packages\transformers\safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\one one\Desktop\bible_reading_recap_nlp\venv\Lib\site-packages\transformers\safetensors_conversion.py", line 72, in get_conversion_pr_reference
    spawn_conversion(token, private, model_id)
  File "c:\one one\Desktop\bible_readi

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,F1 Cross Chapter Verse,F1 Single Chapter,F1 Multi,F1 Noisy,F1 Chapter Range,F1 Book Only,F1 Single Verse,F1 Cross Book,F1 Verse Range,F1 None
1,1.869390,0.724783,0.542443,0.666667,0.598174,0.751735,1.000000,0.578300,0.622200,0.326800,0.902800,0.526300,0.000000,0.733300,0.777800,0.000000
2,0.998685,0.415550,0.849188,0.931298,0.888350,0.951415,1.000000,0.935100,0.877900,0.865700,0.971400,0.898000,1.000000,0.929800,0.947400,0.000000
3,0.727321,0.359093,0.935644,0.961832,0.948557,0.980246,1.000000,1.000000,0.944400,0.937500,0.992800,0.938800,1.000000,0.991000,0.947400,0.000000
4,0.646167,0.323765,0.969697,0.977099,0.973384,0.990390,1.000000,1.000000,0.956500,0.984400,1.000000,0.960000,1.000000,1.000000,1.000000,0.000000
5,0.633616,0.320626,0.964646,0.972010,0.968314,0.987186,1.000000,1.000000,0.956500,0.968800,1.000000,0.960000,1.000000,0.981800,1.000000,0.000000
6,0.629805,0.319419,0.972081,0.974555,0.973316,0.990390,1.000000,1.000000,0.956500,0.976700,1.000000,0.958300,1.000000,1.000000,1.000000,0.000000
7,0.603295,0.315068,0.972081,0.974555,0.973316,0.990924,1.000000,1.000000,0.956500,0.984400,1.000000,0.958300,1.000000,1.000000,1.000000,0.000000
8,0.596269,0.314375,0.974747,0.982188,0.978454,0.991991,1.000000,1.000000,0.968500,0.976700,1.000000,0.979600,1.000000,1.000000,1.000000,0.000000
9,0.593249,0.318225,0.962312,0.974555,0.968394,0.989856,1.000000,1.000000,0.968500,0.930200,1.000000,0.979600,1.000000,1.000000,1.000000,0.000000
10,0.582519,0.316677,0.967337,0.979644,0.973451,0.990924,1.000000,1.000000,0.968500,0.953100,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.71it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

TrainOutput(global_step=1118, training_loss=0.7731845391671005, metrics={'train_runtime': 274.2612, 'train_samples_per_second': 149.201, 'train_steps_per_second': 9.407, 'total_flos': 184242417160104.0, 'train_loss': 0.7731845391671005, 'epoch': 13.0})

## 5. Evaluation & Save

### 5.1 Final Metrics

In [26]:
metrics = trainer.evaluate()

print(f'Experiment     : {EXPERIMENT_NAME}')
print(f'Samples        : {len(sentences)}')
print(f'─' * 40)
print(f'Eval F1        : {metrics["eval_f1"]:.4f}')
print(f'Eval Precision : {metrics["eval_precision"]:.4f}')
print(f'Eval Recall    : {metrics["eval_recall"]:.4f}')
print(f'Eval Accuracy  : {metrics["eval_accuracy"]:.4f}')

Experiment     : indobert-bible-ner-v6
Samples        : 1705
────────────────────────────────────────
Eval F1        : 0.9785
Eval Precision : 0.9747
Eval Recall    : 0.9822
Eval Accuracy  : 0.9920


In [27]:
pattern_metrics = {
    k.replace('eval_f1_', ''): v
    for k, v in metrics.items()
    if k.startswith('eval_f1_')
}

if pattern_metrics:
    # Order by F1 ascending so weakest patterns surface first
    all_patterns  = sorted(set(eval_patterns))
    scored = [(p, pattern_metrics.get(p)) for p in all_patterns]
    scored_only = sorted([(p, f1) for p, f1 in scored if f1 is not None], key=lambda x: x[1])
    skipped = [(p, None) for p, f1 in scored if f1 is None]

    print(f'\n{"Pattern":<22}  {"F1":>6}  {"N eval":>7}')
    print(f'─' * 40)
    for pat, f1 in scored_only + skipped:
        n = sum(1 for p in eval_patterns if p == pat)
        if f1 is None:
            print(f'  {pat:<20}  {"—":>6}  {n:>6}  (skipped — too few samples)')
        else:
            bar = '█' * int(f1 * 20)
            print(f'  {pat:<20}  {f1:>6.4f}  {n:>6}  {bar}')


Pattern                     F1   N eval
────────────────────────────────────────
  none                  0.0000      21  
  multi                 0.9685      56  ███████████████████
  noisy                 0.9767      63  ███████████████████
  book_only             0.9796      25  ███████████████████
  chapter_range         1.0000      69  ████████████████████
  cross_book            1.0000      55  ████████████████████
  cross_chapter_verse   1.0000       3  ████████████████████
  single_chapter        1.0000      38  ████████████████████
  single_verse          1.0000       2  ████████████████████
  verse_range           1.0000       9  ████████████████████


In [28]:
# ── Get raw predictions on eval set ─────────────────────────────────────────
predictions_output = trainer.predict(tokenized_dataset["eval"])
raw_logits = predictions_output.predictions
pred_ids = np.argmax(raw_logits, axis=-1)
gold_ids = predictions_output.label_ids  # -100 for subword continuations

# Decode back to label strings, skipping -100
true_seqs, pred_seqs = [], []
for pred_seq, gold_seq in zip(pred_ids, gold_ids):
    true_row, pred_row = [], []
    for p, g in zip(pred_seq, gold_seq):
        if g == -100:
            continue
        true_row.append(id2label[g])
        pred_row.append(id2label[p])
    true_seqs.append(true_row)
    pred_seqs.append(pred_row)

In [29]:
from collections import defaultdict

# ── Token-level confusion per pattern ───────────────────────────────────────
# error_buckets[pattern] = list of (gold_label, pred_label) pairs
error_buckets = defaultdict(list)

for true_row, pred_row, pattern in zip(true_seqs, pred_seqs, eval_patterns):
    for g, p in zip(true_row, pred_row):
        error_buckets[pattern].append((g, p))

# ── Summary table ────────────────────────────────────────────────────────────
print(f'{"Pattern":<22}  {"Tokens":>6}  {"Correct":>7}  {"Acc":>5}  {"FP":>4}  {"FN":>4}  {"Wrong tag":>9}')
print('─' * 70)

error_records = []

for pat in sorted(set(eval_patterns)):
    pairs = error_buckets[pat]
    total  = len(pairs)
    correct = sum(1 for g, p in pairs if g == p)

    # False Positive: model says entity, gold says O
    fp = sum(1 for g, p in pairs if g == 'O' and p != 'O')
    # False Negative: gold says entity, model says O
    fn = sum(1 for g, p in pairs if g != 'O' and p == 'O')
    # Wrong tag: both non-O but disagree (B vs I confusion)
    wrong_tag = sum(1 for g, p in pairs if g != 'O' and p != 'O' and g != p)

    acc = correct / total if total else 0
    print(f'  {pat:<20}  {total:>6}  {correct:>7}  {acc:>5.3f}  {fp:>4}  {fn:>4}  {wrong_tag:>9}')

    error_records.append({
        'pattern': pat, 'total': total, 'correct': correct,
        'acc': acc, 'fp': fp, 'fn': fn, 'wrong_tag': wrong_tag,
    })

Pattern                 Tokens  Correct    Acc    FP    FN  Wrong tag
──────────────────────────────────────────────────────────────────────
  book_only                 55       54  0.982     0     1          0
  chapter_range            245      245  1.000     0     0          0
  cross_book               301      301  1.000     0     0          0
  cross_chapter_verse        9        9  1.000     0     0          0
  multi                    580      575  0.991     2     1          2
  noisy                    364      362  0.995     2     0          0
  none                     135      128  0.948     7     0          0
  single_chapter           137      137  1.000     0     0          0
  single_verse              13       13  1.000     0     0          0
  verse_range               34       34  1.000     0     0          0


In [30]:
# ── Show the worst sentences per pattern ────────────────────────────────────
# Reconstruct token list from eval split (stored before tokenization removed it)
eval_tokens = [eval_sent[i] for i in range(len(eval_sent))]

print('\n── Failing sentences by pattern ────────────────────────────────────────\n')

for pat in sorted(set(eval_patterns)):
    failures = []
    for i, (true_row, pred_row, pattern) in enumerate(zip(true_seqs, pred_seqs, eval_patterns)):
        if pattern != pat:
            continue
        if true_row == pred_row:
            continue  # correct, skip

        tokens = eval_tokens[i]
        # Align tokens to non-subword positions (same length as true_row)
        annotated = []
        for tok, g, p in zip(tokens, true_row, pred_row):
            mark = '' if g == p else f'[gold={g} pred={p}]'
            annotated.append(f'{tok}{mark}')

        failures.append(' '.join(annotated))

    if not failures:
        print(f'[{pat}] ✓ no errors\n')
    else:
        print(f'[{pat}] {len(failures)} failing sentences:')
        for sent in failures[:5]:   # cap at 5 per pattern
            print(f'  → {sent}')
        if len(failures) > 5:
            print(f'  ... and {len(failures) - 5} more')


── Failing sentences by pattern ────────────────────────────────────────

[book_only] 1 failing sentences:
  → 35 Tiurma Saragih done yojanes[gold=B-BIBLE_REF pred=O]
[chapter_range] ✓ no errors

[cross_book] ✓ no errors

[cross_chapter_verse] ✓ no errors

[multi] 3 failing sentences:
  → Yeh 48 -[gold=O pred=I-BIBLE_REF] Dan[gold=B-BIBLE_REF pred=I-BIBLE_REF] 1 done
  → Anin,2[gold=B-BIBLE_REF pred=O] Raj 8-9 2 Raj 10-11 2 Raj 12-13 done
  → Mazmur 146-147 ; 148-149 ; 150- Amsal[gold=B-BIBLE_REF pred=I-BIBLE_REF] 1 ;[gold=O pred=I-BIBLE_REF] Amsal 2-3 done
[noisy] 2 failing sentences:
  → 25. Liliana dome[gold=O pred=B-BIBLE_REF] 1 kor 3 - 5 🙏
  → Yohanes ✅[gold=O pred=I-BIBLE_REF]
[none] 4 failing sentences:
  → Lukas[gold=O pred=B-BIBLE_REF] lanjut lagi
  → Yohanes[gold=O pred=B-BIBLE_REF] belum selesai
  → Markus[gold=O pred=B-BIBLE_REF] dan Lukas[gold=O pred=I-BIBLE_REF] lagi meeting
  → buka[gold=O pred=B-BIBLE_REF] hal[gold=O pred=I-BIBLE_REF] 141[gold=O pred=I-BIBLE_REF]
[sing

### 5.2 Save Model

In [32]:
SAVE_PATH.mkdir(parents=True, exist_ok=True)
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Model saved to: {SAVE_PATH}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.10s/it]

Model saved to: C:\one one\Desktop\bible_reading_recap_nlp\models\indobert-bible-ner-v6


## 6. Inference

Sanity-check the saved model on a fixed set of test sentences.
These cover the main patterns and known edge cases encountered during development.

### 6.1 Loaded Saved Model

In [38]:
SAVE_PATH = MODEL_DIR / EXPERIMENT_NAME
print(SAVE_PATH)

C:\one one\Desktop\bible_reading_recap_nlp\models\indobert-bible-ner-v6


In [39]:
infer_model, infer_tokenizer = load_ner_model(CONFIG_PATH, SAVE_PATH)
ner_pipeline = pipeline(
    task='ner',
    model=infer_model,
    tokenizer=infer_tokenizer,
    aggregation_strategy='simple',
)

print('Inference pipeline ready.')

2026-04-05 10:20:18 | INFO     | bible_pipeline.extraction.extractor | Loading model from saved path: C:\one one\Desktop\bible_reading_recap_nlp\models\indobert-bible-ner-v6


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 413.50it/s, Materializing param=classifier.weight]                                      


Inference pipeline ready.


### 6.2 Raw NER Output

In [42]:
test_sentences = [
    # Standard patterns 
    'Efesus 1-2 done',
    'Why 1-2 done',
    'Efs1-2 done',                        
    '_Ef 1-2_ ✓',                          
    'Gal 3-6 done',
    '2 Kor 4-9🙏',
    '2 Pet 11-13 ☑️',
    '1 Korintus 14 - 15 selesai.',
    'Kis 1-10 done',
    'Kis para rasul 1-10 done',

    # Cross-book ranges
    '1 Korintus 16 - 2 Korintus 1 selesai',
    'Gal 6- Ef 2 done',
    'Ibrani 11-Yakobus 1; 2-3; 4-5',
    'Kel 40-Im 2 done',
    'Bil 36 - Ul 2 selesai',

    # Comma / list patterns 
    '1 Kor 1,2,3,4,5 done',
    'Mazmur 1, 23, 91 selesai',
    'Mat 5, 6, 7 done',
    'Kej 1,3,5,7 selesai',

    # Minor Prophets comma list 
    'Amos, Obaja, Yunus, Mikha, Nahum, Habakuk, Zefanya, Hagai, Zakaria, Maleaki done',

    # Multiline 
    'Kej 1-3 done Kej 4-6 done Kej 7-9 done',
    'Dan 1-2 done Dan 3-4 done',
    '3 Yohanes ✅ Yudas ✅',

    # Compound book names 
    'hakim-hakim 6 - hakim-hakim 7 selesai',
    '```Kidung Agung 5 - Kidung Agung 6```✅',
    '1 Raja-raja 3-5 done',

    # Book-range prefix
    '1-3 Yoh done',
    '1-2 Kor done',
    '1-2 Samuel selesai',

    # Semi-colon separated
    '1 petrus done; 2 petrus done; 3 petrus done',
    'Kej 1-10 done; Kel 1-5 done',
    'Kej 5-6 done 7-8 done 9-10 done',

    # Verse-level / mixed formats
    'I Sam 2:1-5',
    'Mat 6:9-13 selesai',
    'Yoh 3:16',
    'Roma 8:28 done',

    # Noisy / real-life progress text
    "17. Jason done kej 1-30",
    'Saya sudah sampai Kej 36, terima kasih',
    'Bu Dessi, saya belum selesai baca Daniel 13',
    'Progress hari ini: Kel 1-12 selesai 🙏',

    # Hard negatives / ambiguity
    'Besok daniel ulang tahun',
    'Jojo daniel 1-3 done',
    'Zed 1-3 done',
    'Raja film 2 bagus banget',

    # Unicode / dash variations
    'Gal 6—Ef 2 done',
    'Ef 1–3 selesai',
    'Kej 1—3 done',
    'Kej 1 s/d 2 done',
    ' Mikha 6/7 done',

    # Messy / OCR-like / spacing issues
    '2 Raja - blraja 6 - 7 done',
    '1kor1-3 done',
    'kej1 - 5done',
    'Maz  23  done',

    # Long chained sequences
    '1 Raja-raja 20-21 done 1 Raja-raja 22 - 2 Raja-raja 1 done 2 Raja-raja 2-3 done',
    
    # Book only
    'Mazmur done',
    'Kejadian selesai',
    'Wahyu ✓',
    "1 Raja dan 2 Raja",

    # experimental cases
    "Jonathan kej 1-3 done",
    "Jessica ams 5-10 done",
    "Jeffry Kejadian 7-8 done Jessica Kejadian 7-8 done",
    "**Bible Reading Cycle 2, Sabtu, 17 Okt 2020** Renungan: Pasal ini mengingatkan kita kembali pd anugerah Tuhan di tengah dosa pemberontakan yg Israel lakukan. Hal ini jg mengingatkan kita pd perkataan Petrus dlm 2Ptr 3:8-9, kalau Tuhan masih begitu bersabar thd dunia yg berdosa ini dikrnkan Dia tdk menghendaki ada yg binasa. Pernahkan kita melakukan dosa yg tdk mendatangkan hukuman Tuhan? Apakah kita boleh bersombong di dlm dosa tsb dikrnkan tdk ada intervensi Tuhan? Bgmanakah kita seharusnya bersikap berdasarkan perikop ini? Enjoy your bible reading ☕📖🙏"
]

print('Raw NER output (aggregation_strategy=simple):')
for sentence in test_sentences:
    results = ner_pipeline(sentence)
    print(f'\nInput : {sentence}')
    for r in results:
        print(f"  {r['entity_group']:12s} | {r['word']:20s} | score: {r['score']:.4f}")
        if r['entity_group'] == 'BIBLE_REF':
            parsed = parse_ner_response(r['word'])
            normalized = normalizer.normalize(parsed)
            for ref in normalized:
                print(f'  →  {ref}')

Raw NER output (aggregation_strategy=simple):

Input : Efesus 1-2 done
  BIBLE_REF    | efesus 1 - 2         | score: 0.9433
  →  {'book_start': 'Efesus', 'start_chapter': 1, 'book_end': 'Efesus', 'end_chapter': 2, 'is_valid': True}

Input : Why 1-2 done
  BIBLE_REF    | why 1 - 2            | score: 0.9440
  →  {'book_start': 'Wahyu', 'start_chapter': 1, 'book_end': 'Wahyu', 'end_chapter': 2, 'is_valid': True}

Input : Efs1-2 done
  BIBLE_REF    | efs1 - 2             | score: 0.9461
  →  {'book_start': 'Efesus', 'start_chapter': 1, 'book_end': 'Efesus', 'end_chapter': 2, 'is_valid': True}

Input : _Ef 1-2_ ✓
  BIBLE_REF    | ef 1 - 2             | score: 0.9450
  →  {'book_start': 'Efesus', 'start_chapter': 1, 'book_end': 'Efesus', 'end_chapter': 2, 'is_valid': True}

Input : Gal 3-6 done
  BIBLE_REF    | gal 3 - 6            | score: 0.9439
  →  {'book_start': 'Galatia', 'start_chapter': 3, 'book_end': 'Galatia', 'end_chapter': 6, 'is_valid': True}

Input : 2 Kor 4-9🙏
  BIBLE_REF   